# ECE 343 Lab: Magnetic Fields in Python and on the Bench (Lesson 32)

<a href="https://colab.research.google.com/github/livethisdream/ece343/blob/main/book/extras/labs/ECE343_Lab_L32_Magnetostatics.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Team members:**

**Documentation:**

This lab computes the magnetic fields of Module 3 and then measures one. Part A
builds fields from the Biot-Savart law (Lesson 28) by adding up short pieces of
current, and checks each one against a result you derived by hand: the finite
and infinite wire (Lessons 28 and 29), the loop on its axis (Lesson 28), and the
solenoid (Lesson 30). It also tests Gauss's law for magnetism and Ampere's law on
a grid. Part B measures the field on the axis of a coil you wind, with a linear
Hall sensor, and compares it with Part A.

**Teams.** Work in a team of two or three. Each team submits one report.

**Deliverable.** This notebook, run top to bottom with every code cell
complete and every **Answer** cell filled in, exported to PDF and uploaded to
Gradescope. Put every member's name in the cell above, and write a
Documentation line (**None** if you did not collaborate outside your team).

**Running it.** Open it in Google Colab with the badge above, or run it
locally with Python 3, NumPy, and Matplotlib. Run the cells in order. A cell
marked `TODO` is yours to finish; the rest are given.

**Reference.** Ellingson, *Electromagnetics*, Vol. 1, Chapter 7, and the
Lesson 28 through 30 pages on the course site.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

MU0 = 4e-7 * np.pi                # permeability of free space (H/m)


def biot_savart(path, I, pts):
    """H (A/m) at the points pts from current I (A) along a polyline.

    path: (M + 1, 3) array of vertices in meters; the current runs from the
          first vertex to the last. Close a loop by repeating the first vertex.
    pts:  (..., 3) array of field points in meters.
    Each straight piece dl, centered at its midpoint, adds
    dH = I dl x R / (4 pi R^3), Lesson 28's Biot-Savart law.
    """
    path = np.asarray(path, float)
    pts = np.asarray(pts, float)
    dl = np.diff(path, axis=0)                    # (M, 3) current elements
    mid = 0.5 * (path[:-1] + path[1:])            # (M, 3) their centers
    H = np.zeros_like(pts)
    for d, m in zip(dl, mid):                     # loop over pieces, not points
        R = pts - m
        Rmag = np.linalg.norm(R, axis=-1, keepdims=True)
        H += np.cross(d, R) / Rmag**3
    return I / (4 * np.pi) * H


def line_path(start, end, M):
    """A straight wire from start to end, split into M pieces."""
    t = np.linspace(0, 1, M + 1)[:, None]
    return (1 - t) * np.asarray(start, float) + t * np.asarray(end, float)


def axis_points(z):
    """Points on the z axis, as an (n, 3) array."""
    z = np.atleast_1d(z)
    return np.stack([np.zeros_like(z), np.zeros_like(z), z], axis=-1)

---
## Pre-Lab: Predictions

Work these by hand before the lab period, from Lessons 28 through 30. Part B uses
a flat coil of $N = 100$ turns and radius $a = 2.0\ \text{cm}$ carrying
$I = 1.0\ \text{A}$, and a linear Hall sensor whose output rises by
$S = 14\ \text{mV}$ for each millitesla (SS49E) or $50\ \text{mV/mT}$ (A1324).

1. $B = \mu_0 H$ at the center of the coil, treating its 100 turns as one loop of current $NI$.
2. $B$ on the axis at $z = 2.0\ \text{cm}$ and $z = 4.0\ \text{cm}$.
3. The change in sensor voltage at the center, for your sensor.
4. The resistance of the coil wound from 26 AWG magnet wire ($0.134\ \Omega/\text{m}$), and the power it dissipates at $1.0\ \text{A}$.

**Answer:**

---
## Part A: The Biot-Savart Law on a Computer

The function `biot_savart` above is Lesson 28's law with the integral replaced
by a sum: it splits a wire into $M$ straight pieces, puts each piece's current
element $I\ d\mathbf{l}$ at the piece's center, and adds the contributions

$$\mathbf{H} \approx \sum_{k=1}^{M} \frac{I\ \Delta\mathbf{l}_k\times\mathbf{R}_k}{4\pi R_k^3}.$$

As $M$ grows, the sum approaches the integral. NumPy evaluates each piece's
contribution at every field point at once, so the only loop is over the pieces.

### A1. A finite wire, and the infinite-wire limit

A wire of length $L = 1.0\ \text{m}$ lies on the $z$ axis from $-L/2$ to
$L/2$ and carries $I = 1.0\ \text{A}$ in $+\hat{\mathbf{z}}$. Lesson 28 found
the field in the plane $z = 0$:

$$\mathbf{H} = \hat{\boldsymbol{\phi}}\ \frac{I L}{4\pi\rho\sqrt{\rho^2 + L^2/4}},$$

which approaches Lesson 29's infinite-wire result, $I/(2\pi\rho)$, when
$L \gg \rho$. Finish the two `TODO` lines, then run the cell.

In [ ]:
I, L = 1.0, 1.0
wire = line_path([0, 0, -L / 2], [0, 0, L / 2], M=400)

rho = np.linspace(0.01, 0.5, 50)                       # m, along +x in z = 0
pts = np.stack([rho, np.zeros_like(rho), np.zeros_like(rho)], axis=-1)
H_num = biot_savart(wire, I, pts)[:, 1]                # H_phi is H_y on the +x axis

H_finite = np.zeros_like(rho)    # TODO: Lesson 28's finite-wire formula
H_inf = np.zeros_like(rho)       # TODO: Lesson 29's infinite-wire formula

plt.figure(figsize=(6, 4))
plt.loglog(rho, H_num, "o", ms=4, label="Biot-Savart sum")
plt.loglog(rho, H_finite, label="finite wire, Lesson 28")
plt.loglog(rho, H_inf, "--", label="infinite wire, Lesson 29")
plt.xlabel("ρ (m)"); plt.ylabel("H (A/m)"); plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.show()
print("largest relative difference, sum vs Lesson 28:",
      np.max(np.abs(H_num - H_finite) / H_finite))

In [ ]:
# How many pieces does the sum need? The field 5 cm from the wire's center.
p = np.array([[0.05, 0.0, 0.0]])
exact = I * L / (4 * np.pi * 0.05 * np.sqrt(0.05**2 + L**2 / 4))
for M in (4, 10, 20, 40, 400):
    Hy = biot_savart(line_path([0, 0, -L / 2], [0, 0, L / 2], M), I, p)[0, 1]
    print(f"M = {M:4d}:  H = {Hy:.4f} A/m,  error {100 * (Hy - exact) / exact:+.2f}%")

**Question A1.** (a) At what distance $\rho$ does the 1 m wire's field fall 2%
below the infinite-wire value? Work it from the two formulas, and check it on
your plot. (b) How many pieces does the sum need for 0.1% accuracy 5 cm from
the wire, and why does it need more pieces close to the wire than far from it?

**Answer:**

### A2. A loop on its axis

Lesson 28 integrated the Biot-Savart law around a circular loop of radius $a$
in the plane $z = 0$ and found, on the axis,

$$\mathbf{H} = \hat{\mathbf{z}}\ \frac{I a^2}{2\left(a^2 + z^2\right)^{3/2}}.$$

Write `loop_path`, which returns the vertices of an $M$-sided polygon of radius
$a$ in the plane $z = z_0$, traversed counterclockwise seen from $+z$, with the
first vertex repeated at the end so the loop closes.

In [ ]:
def loop_path(a, M, z0=0.0):
    """An M-piece circular loop of radius a in the plane z = z0, closed."""
    phi = np.linspace(0, 2 * np.pi, M + 1)
    # TODO: return an (M + 1, 3) array of [x, y, z] vertices
    return np.zeros((M + 1, 3))


a, I = 0.02, 1.0
z = np.linspace(-0.08, 0.08, 161)
H_formula = I * a**2 / (2 * (a**2 + z**2) ** 1.5)

plt.figure(figsize=(6, 4))
for M in (8, 32, 128):
    Hz = biot_savart(loop_path(a, M), I, axis_points(z))[:, 2]
    print(f"M = {M:3d}: H at the center = {Hz[80]:.3f} A/m "
          f"(Lesson 28: {H_formula[80]:.3f} A/m)")
    plt.plot(100 * z, Hz, label=f"sum, M = {M}")
plt.plot(100 * z, H_formula, "k--", label="Lesson 28")
plt.xlabel("z (cm)"); plt.ylabel("H_z (A/m)"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

**Question A2.** (a) Does the polygon's field at the center come out above or
below Lesson 28's value, and by how much for $M = 8$? Explain the sign from the
shape of the polygon. (b) Far from the loop, how does $H_z$ fall with $z$, and
what does that tell you about the loop seen from far away (Lesson 28)?

**Answer:**

### A3. Field lines of a loop, and a Helmholtz pair

The next cell computes $\mathbf{H}$ of the loop in the $x$–$z$ plane, which
contains the axis, and draws its field lines. Then it places two identical
loops on a common axis a distance $d$ apart, both carrying $I$ in the same
direction. With $d = a$, the pair is a **Helmholtz pair**: the field near the
middle is unusually uniform, which is why lab coils for calibrating magnetic
sensors are built this way.

In [ ]:
a, I = 0.02, 1.0
g = np.linspace(-0.05, 0.05, 121)
Xg, Zg = np.meshgrid(g, g)
grid = np.stack([Xg, np.zeros_like(Xg), Zg], axis=-1)
Hg = biot_savart(loop_path(a, 128), I, grid)

fig, ax = plt.subplots(figsize=(6, 6))
ax.streamplot(Xg * 100, Zg * 100, Hg[..., 0], Hg[..., 2], density=1.4,
              color=np.log10(np.hypot(Hg[..., 0], Hg[..., 2])), cmap="viridis")
ax.plot([-a * 100, a * 100], [0, 0], "o", color="tab:red", ms=6)   # where the wire crosses the plane
ax.set_aspect("equal"); ax.set_xlabel("x (cm)"); ax.set_ylabel("z (cm)")
ax.set_title("Field lines of one loop in the x–z plane")
plt.show()

In [ ]:
def pair_axis_field(a, d, I, z, M=128):
    """H_z on the axis of two coaxial loops at z = -d/2 and z = +d/2."""
    # TODO: add the fields of the two loops
    return np.zeros_like(z)


z = np.linspace(-0.04, 0.04, 401)
plt.figure(figsize=(6, 4))
for d, lab in ((0.5 * a, "d = a/2"), (a, "d = a (Helmholtz)"), (2 * a, "d = 2a")):
    Hz = pair_axis_field(a, d, I, z)
    plt.plot(100 * z, Hz / Hz[200], label=lab)
plt.ylim(0.8, 1.1)
plt.xlabel("z (cm)"); plt.ylabel("H_z / H_z(0)"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

Hh = pair_axis_field(a, a, I, z)
flat = np.abs(Hh / Hh[200] - 1) < 0.01
print(f"Helmholtz pair: H_z(0) = {Hh[200]:.2f} A/m; within 1% of it for "
      f"|z| < {100 * np.max(np.abs(z[flat])):.2f} cm")

**Question A3.** (a) In the field-line plot, do any lines start or end? Where
do the lines through the middle of the loop go? (b) Describe how the field
near the center changes as the spacing goes from $a/2$ to $a$ to $2a$, and
explain why there is a spacing in between at which it is flattest. (c) Over
what range of $z$ is the Helmholtz pair's field within 1% of its center value?

**Answer:**

### A4. A solenoid of finite length

Lesson 30 used Ampere's law to find $H = nI$ inside a long solenoid, with
$n = N/\ell$ turns per meter, and then added up loops to find the field at the
center of a finite one:

$$H_z(0) = nI\ \frac{\ell}{\sqrt{\ell^2 + 4a^2}}.$$

Build a solenoid as a stack of $N$ loops spread evenly from $z = -\ell/2$ to
$z = +\ell/2$, and compute its field at the center for several lengths at the
same $n$.

In [ ]:
def solenoid_axis_field(a, ell, n, I, z, M=48):
    """H_z on the axis of a solenoid built as a stack of loops."""
    N = max(int(round(n * ell)), 1)
    z_turns = np.linspace(-ell / 2, ell / 2, N)
    # TODO: add the field of a loop at each position in z_turns
    return np.zeros_like(z)


a, n, I = 0.02, 2000.0, 1.0          # 2000 turns per meter
print(" ell/a   center/nI (stack)   center/nI (Lesson 30)")
for ratio in (1, 2, 4, 10, 20):
    ell = ratio * a
    Hc = solenoid_axis_field(a, ell, n, I, np.array([0.0]))[0]
    print(f"{ratio:5d}   {Hc / (n * I):17.3f}   {ell / np.sqrt(ell**2 + 4 * a**2):19.3f}")

In [ ]:
# The field along the axis of the Part B solenoid: 200 turns over 10 cm.
ell = 0.10
z = np.linspace(-0.08, 0.08, 161)
Hs = solenoid_axis_field(a, ell, n, I, z)
plt.figure(figsize=(6, 4))
plt.plot(100 * z, Hs / (n * I))
plt.axhline(1, color="k", ls="--", lw=1)
plt.axvline(-5, color="gray", lw=0.8); plt.axvline(5, color="gray", lw=0.8)
plt.xlabel("z (cm)"); plt.ylabel("H_z / nI"); plt.grid(alpha=0.3)
plt.title("Solenoid, ℓ = 10 cm, a = 2 cm (gray lines: its ends)")
plt.show()
print(f"center: {Hs[80] / (n * I):.3f} nI;  end (z = 5 cm): {Hs[130] / (n * I):.3f} nI")

**Question A4.** (a) How long must the solenoid be, in units of $a$, for its
center field to be within 2% of $nI$? (b) Lesson 30 found that the field at
either end of a long solenoid is $nI/2$. Explain that result from the stack of
loops, using superposition and no integral.

**Answer:**

### A5. Gauss's law for magnetism and Ampere's law on a grid

Two long straight wires run parallel to the $z$ axis: one carries $3\ \text{A}$
in $+\hat{\mathbf{z}}$ at $(-0.1, 0)\ \text{m}$, and the other $2\ \text{A}$ in
$-\hat{\mathbf{z}}$ at $(0.1, 0)\ \text{m}$. Each has Lesson 29's field
$I/(2\pi\rho)$ around it. A long wire's field is the same in every plane of
constant $z$, so $\partial/\partial z = 0$ and a grid in the $x$–$y$ plane holds
the whole field. That makes two tests exact:

- **Point form.** $\nabla\cdot\mathbf{B} = \partial B_x/\partial x + \partial B_y/\partial y$
  should be zero everywhere except on the wires.
- **Integral forms, per meter of length.** Around a closed contour $C$ in the
  plane, Gauss's law for magnetism makes the flux of $\mathbf{B}$ out through
  $C$ zero, and Ampere's law makes the circulation $\oint_C \mathbf{H}\cdot d\mathbf{l}$
  equal to $I_\text{encl}$.

In [ ]:
wires = [(-0.1, 0.0, 3.0), (0.1, 0.0, -2.0)]       # (x, y, I in +z)

x = np.linspace(-0.5, 0.5, 400)                      # even count keeps points off the wires
h = x[1] - x[0]
X, Y = np.meshgrid(x, x)


def wire_field(X, Y, wires):
    """H (A/m) of infinite wires along z: H = phi_hat I / (2 pi rho), Lesson 29."""
    Hx, Hy = np.zeros_like(X), np.zeros_like(X)
    for xw, yw, Iw in wires:
        dx, dy = X - xw, Y - yw
        r2 = dx**2 + dy**2
        Hx += -Iw * dy / (2 * np.pi * r2)            # phi_hat = (-dy, dx)/rho
        Hy += Iw * dx / (2 * np.pi * r2)
    return Hx, Hy


Hx, Hy = wire_field(X, Y, wires)
Bx, By = MU0 * Hx, MU0 * Hy

fig, ax = plt.subplots(figsize=(6, 6))
ax.streamplot(X, Y, Hx, Hy, density=1.4, color=np.log10(np.hypot(Hx, Hy)), cmap="viridis")
for xw, yw, Iw in wires:
    ax.plot(xw, yw, "o", color="tab:red" if Iw > 0 else "tab:blue", ms=6)
ax.set_aspect("equal"); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_title("3 A out of the page (red), 2 A into it (blue)")
plt.show()

In [ ]:
# Point form: the two-term divergence, by centered differences.
divB = np.gradient(Bx, h, axis=1) + np.gradient(By, h, axis=0)

# A single derivative of B is about |B|/d, where d is the distance to the
# nearest wire. Dividing by that scale makes the test dimensionless: a value
# near 1 would mean the terms do not cancel; near 0 means they do.
d = np.min([np.hypot(X - xw, Y - yw) for xw, yw, _ in wires], axis=0)
ratio = np.abs(divB) * d / np.hypot(Bx, By)
use = d > 10 * h                               # stay off the wires
use[[0, -1], :] = use[:, [0, -1]] = False      # and off the grid's edges
print(f"|div B| / (|B|/d), median over the grid: {np.median(ratio[use]):.1e}")

In [ ]:
def square_contour(xc, yc, s, npts=4000):
    """Points around a square of side s centered at (xc, yc), counterclockwise,
    with the unit outward normal and the step length at each point."""
    t = (np.arange(npts) + 0.5) / npts * 4
    side, u = np.floor(t).astype(int), t % 1
    px = np.select([side == 0, side == 1, side == 2, side == 3],
                   [-s / 2 + u * s, s / 2 * np.ones_like(u), s / 2 - u * s, -s / 2 * np.ones_like(u)])
    py = np.select([side == 0, side == 1, side == 2, side == 3],
                   [-s / 2 * np.ones_like(u), -s / 2 + u * s, s / 2 * np.ones_like(u), s / 2 - u * s])
    nx = np.select([side == 1, side == 3], [1.0, -1.0], 0.0)
    ny = np.select([side == 0, side == 2], [-1.0, 1.0], 0.0)
    return xc + px, yc + py, nx, ny, 4 * s / npts


def flux_and_circulation(xc, yc, s):
    px, py, nx, ny, dl = square_contour(xc, yc, s)
    hx, hy = wire_field(px, py, wires)
    flux = np.sum(MU0 * (hx * nx + hy * ny)) * dl     # Wb per meter of length
    # TODO: the circulation of H. The unit tangent, counterclockwise, is (-ny, nx).
    circ = 0.0
    return flux, circ


for xc, yc, s, what in [(-0.1, 0.0, 0.1, "around the 3 A wire"),
                        (0.1, 0.0, 0.1, "around the 2 A wire"),
                        (0.0, 0.0, 0.5, "around both"),
                        (0.0, 0.3, 0.2, "around neither")]:
    flux, circ = flux_and_circulation(xc, yc, s)
    print(f"{what:20s}: flux of B = {flux:+.2e} Wb/m,  circulation of H = {circ:+.4f} A")

**Question A5.** (a) Is $\nabla\cdot\mathbf{B}$ zero away from the wires? Compare
it with the size of a single derivative to judge. (b) Explain each of the four
circulations with Ampere's law, including its sign. (c) The flux of
$\mathbf{B}$ out through every square is zero, even the squares with a wire
inside. Compare with Lesson 26, where the flux of $\epsilon_0\mathbf{E}$
through a square around a line charge was the charge inside. What is the
difference, in one sentence?

**Answer:**

**Question A6 (the loop's cross section).** Repeat the divergence test on the
loop's field from A3, in the $x$–$z$ plane. The two-term sum
$\partial H_x/\partial x + \partial H_z/\partial z$ is not zero. Explain why,
find the missing term, and show that adding it makes the divergence vanish.
Hint: in that plane $H_y = 0$, but the loop's field is symmetric about its
axis, so $\partial H_y/\partial y$ at a point on the plane equals $H_x/x$
(Lesson 17's divergence in cylindrical coordinates).

In [ ]:
gx = g[1] - g[0]
two_term = np.gradient(Hg[..., 0], gx, axis=1) + np.gradient(Hg[..., 2], gx, axis=0)
missing = np.zeros_like(two_term)     # TODO: the missing term, H_x / x (careful at x = 0)

# The same dimensionless test as A5: divide by |H|/d, d the distance to the wire.
d_w = np.hypot(np.abs(Xg) - a, Zg)
use = (d_w > 8 * gx) & (np.abs(Xg) > 2 * gx)
Hmag = np.hypot(Hg[..., 0], Hg[..., 2])
print(f"two-term sum / (|H|/d), median:       {np.median((np.abs(two_term) * d_w / Hmag)[use]):.1e}")
print(f"with the missing term / (|H|/d), median: {np.median((np.abs(two_term + missing) * d_w / Hmag)[use]):.1e}")

**Answer:**

---
## Part B: The Field of a Coil, Measured

Each team winds a flat coil and measures $B_z$ on its axis with a linear Hall
sensor, then compares the measurement with the Biot-Savart curve from Part A.

**Equipment.** About 13 m of 26 AWG magnet wire; a tube about 4 cm in diameter
(PVC pipe or a cardboard tube); sandpaper; tape; the bench DC supply; two
digital multimeters (one on DC millivolts for the sensor, one on DC amps in
series with the coil, or read the supply's current); a linear Hall sensor
(SS49E or A1324) with a second 5 V output for it; a ruler.

**Safety.** At 1 A the coil dissipates about 1.7 W and warms within a few
minutes. Turn the current on only while you take a reading, and turn the supply
off if the coil is too warm to hold. Set the supply's current limit to 1.0 A
before you connect the coil. Sand the enamel off the last centimeter of each
wire end before you connect it; enamel does not conduct.

### B1. Build and set up

1. Wind $N = 100$ turns tightly on the tube, in a band about 1 cm wide. Tape the turns so they cannot slide, and leave about 30 cm of each end free.
2. Measure the coil's inner and outer diameter with the ruler and record the mean radius $a$ below.
3. Power the sensor from 5 V and connect its output and ground to the millivolt meter. Its output sits near 2.5 V with no field and rises by $S$ for each millitesla along the face marked in its datasheet.
4. Tape the sensor to the ruler, face toward the coil, so that it can slide along the coil's axis. Mark $z = 0$ where the sensor's face is in the plane of the coil.
5. Connect the coil to the supply through the ammeter and set $I = 1.0\ \text{A}$.

### B2. Measure

At each position, record the sensor voltage with the coil off, $V_\text{off}$,
then with it on, $V_\text{on}$. The difference removes the sensor's 2.5 V offset
and Earth's field, which are the same in both readings, so

$$B_z = \frac{V_\text{on} - V_\text{off}}{S}.$$

Take readings from $z = 0$ to $z = 8\ \text{cm}$ in steps of $1\ \text{cm}$.

In [ ]:
# TODO: your measurements. z in cm from the coil's plane; voltages in volts.
N_coil = 100
a_coil = 0.020                 # TODO: your measured mean radius (m)
I_coil = 1.0                   # TODO: your measured current (A)
S = 0.014                      # TODO: your sensor's sensitivity (V/mT): 0.014 SS49E, 0.050 A1324

z_cm = np.arange(0, 9)
V_off = np.full(z_cm.shape, 2.500)     # TODO
V_on = np.full(z_cm.shape, 2.500)      # TODO

In [ ]:
B_meas = (V_on - V_off) / S                                    # mT

z_fine = np.linspace(0, 0.08, 161)
loop = loop_path(a_coil, 128)
B_pred = MU0 * N_coil * biot_savart(loop, I_coil, axis_points(z_fine))[:, 2] * 1e3   # mT

plt.figure(figsize=(6, 4))
plt.semilogy(100 * z_fine, B_pred, label=f"Biot-Savart, a = {100 * a_coil:.2f} cm")
plt.semilogy(z_cm, B_meas, "o", label="measured")
plt.xlabel("z (cm)"); plt.ylabel("B_z (mT)"); plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.show()

B_pred_at = np.interp(z_cm / 100, z_fine, B_pred)
for zc, bm, bp in zip(z_cm, B_meas, B_pred_at):
    print(f"z = {zc} cm: measured {bm:.3f} mT, predicted {bp:.3f} mT, ratio {bm / bp:.3f}")

### B3. Where the differences come from

Two dimensions are hard to know to better than a couple of millimeters: where
the Hall element sits inside the sensor's package, and the coil's effective
radius, since 100 turns of wire make a band with some width and depth. The next
cell tries a range of sensor offsets and coil radii and finds the pair that
best fits your data.

In [ ]:
best = None
for dz in np.arange(-0.004, 0.0041, 0.0005):           # sensor offset (m)
    for aa in np.arange(0.018, 0.0231, 0.0002):         # coil radius (m)
        zz = z_cm / 100 + dz
        model = MU0 * N_coil * I_coil * aa**2 / (2 * (aa**2 + zz**2) ** 1.5) * 1e3
        err = np.sum(np.log(B_meas / model) ** 2)       # compare ratios, not differences
        if best is None or err < best[0]:
            best = (err, dz, aa)
_, dz_fit, a_fit = best
print(f"best fit: sensor offset {1000 * dz_fit:+.1f} mm, coil radius {100 * a_fit:.2f} cm")

**Question B.** (a) Is your measured field above or below the prediction near
the coil, and far from it? (b) What sensor offset and coil radius fit best, and
are they physically reasonable for your coil and sensor? (c) Name one other
source of error, and say how you would reduce it.

**Answer:**

### B4 (optional). A solenoid

If your instructor assigns it, wind 200 turns in a single layer over
$\ell = 10\ \text{cm}$ of the same tube (about 25 m of wire, $3.4\ \Omega$). Run
it at $0.5\ \text{A}$ so it stays cool, and measure $B_z$ at the center and at
each end. Part A4 predicts $0.928\ \mu_0 nI = 1.17\ \text{mT}$ at the center and
$0.490\ \mu_0 nI = 0.62\ \text{mT}$ at the ends, with $n = 2000\ \text{turns/m}$.

In [ ]:
I_sol = 0.5
B_center, B_end = 0.0, 0.0      # TODO: your measured values (mT)
n_sol, a_sol, ell_sol = 2000.0, 0.020, 0.10
pred_c = MU0 * n_sol * I_sol * ell_sol / np.sqrt(ell_sol**2 + 4 * a_sol**2) * 1e3
pred_e = MU0 * n_sol * I_sol * ell_sol / (2 * np.sqrt(ell_sol**2 + a_sol**2)) * 1e3
print(f"center: measured {B_center:.3f} mT, predicted {pred_c:.3f} mT")
print(f"end:    measured {B_end:.3f} mT, predicted {pred_e:.3f} mT")

---
## Report

Before you submit, run the whole notebook from the top (**Runtime**, then
**Run all** in Colab) so every output is current. Then:

1. Check that every `TODO` is done and every **Answer** cell is filled in.
2. Put every team member's name and your Documentation line in the first cell.
3. Export to PDF (**File**, then **Print**, then **Save as PDF**) and upload it to Gradescope as one team submission, adding every member.